In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import pickle
from sklearn.metrics import classification_report, roc_auc_score

In [34]:
df=pd.read_csv('../data/processed_loanDefaulter.csv')

In [35]:
X = df.drop(columns='Default', axis=1)
y=df['Default']

In [36]:
X=df.drop("Default", axis=1)
y=df["Default"]

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)

In [38]:
def evaluate_model(true, predicted, prob):
    print(classification_report(true, predicted))
    print("ROC-AUC Score:", roc_auc_score(true, prob))

In [39]:
scale_pos_weight = 225694 / 29653

models = {
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss')
}

In [40]:

for name, model in models.items():
    
    print(f"\n===== {name} =====")
    
    # Logistic needs scaling
    if name == "Logistic Regression":
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:,1]
    else:
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test )[:,1]
    
    # Default threshold
    y_pred = (y_prob > 0.5).astype(int)
    
    print("\n🔹 Default Threshold (0.5)")
    evaluate_model(y_test, y_pred, y_prob)
    
    # 🔥 Threshold tuning
    for t in [0.4, 0.3]:
        y_pred_tuned = (y_prob > t).astype(int)
        print(f"\n🔹 Threshold = {t}")
        print(classification_report(y_test, y_pred_tuned))
    
    print("="*50)


===== XGBoost =====

🔹 Default Threshold (0.5)
              precision    recall  f1-score   support

           0       0.94      0.73      0.82     45139
           1       0.23      0.63      0.34      5931

    accuracy                           0.72     51070
   macro avg       0.59      0.68      0.58     51070
weighted avg       0.86      0.72      0.76     51070

ROC-AUC Score: 0.7423024622768384

🔹 Threshold = 0.4
              precision    recall  f1-score   support

           0       0.95      0.60      0.73     45139
           1       0.20      0.75      0.31      5931

    accuracy                           0.61     51070
   macro avg       0.57      0.67      0.52     51070
weighted avg       0.86      0.61      0.68     51070


🔹 Threshold = 0.3
              precision    recall  f1-score   support

           0       0.96      0.44      0.60     45139
           1       0.17      0.85      0.28      5931

    accuracy                           0.49     51070
   macro

In [41]:
xgb_params = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 8, 12],
    "subsample": [0.7, 0.8, 1],
    "colsample_bytree": [0.5, 0.7, 1],
    "gamma": [0, 0.1, 0.3],
    "min_child_weight": [1, 3, 5]
}

In [42]:
scale_pos_weight = 225694 / 29653

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False
)

In [43]:
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_params,
    n_iter=25,              # keep balanced (speed vs performance)
    scoring='roc_auc',      # 🔥 VERY IMPORTANT
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print("🚀 Starting XGBoost Hyperparameter Tuning...")
random_search.fit(X_train, y_train)

🚀 Starting XGBoost Hyperparameter Tuning...
Fitting 3 folds for each of 25 candidates, totalling 75 fits


c:\Users\SOHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:08:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.5, 0.7, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], ...}"
,n_iter,25
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [44]:
best_xgb = random_search.best_estimator_

best_xgb.fit(X_train, y_train)

c:\Users\SOHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:08:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [45]:
y_prob = best_xgb.predict_proba(X_test)[:,1]

y_pred = (y_prob > 0.4).astype(int)

print("\n🔥 Final Model Performance (Threshold = 0.4)")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


🔥 Final Model Performance (Threshold = 0.4)
              precision    recall  f1-score   support

           0       0.96      0.54      0.69     45139
           1       0.19      0.82      0.31      5931

    accuracy                           0.57     51070
   macro avg       0.57      0.68      0.50     51070
weighted avg       0.87      0.57      0.65     51070

ROC-AUC: 0.760162622352121


In [ ]:
for t in [0.35, 0.4, 0.45]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold = {t}")
    print(classification_report(y_test, y_pred))  


Threshold = 0.35
              precision    recall  f1-score   support

           0       0.96      0.45      0.61     45139
           1       0.17      0.86      0.29      5931

    accuracy                           0.50     51070
   macro avg       0.57      0.66      0.45     51070
weighted avg       0.87      0.50      0.58     51070


Threshold = 0.4
              precision    recall  f1-score   support

           0       0.96      0.54      0.69     45139
           1       0.19      0.82      0.31      5931

    accuracy                           0.57     51070
   macro avg       0.57      0.68      0.50     51070
weighted avg       0.87      0.57      0.65     51070


Threshold = 0.45
              precision    recall  f1-score   support

           0       0.95      0.62      0.75     45139
           1       0.21      0.76      0.33      5931

    accuracy                           0.64     51070
   macro avg       0.58      0.69      0.54     51070
weighted avg       0.

In [47]:
model_package = {
    "model": best_xgb,
    "threshold": 0.45
}

with open("../artifacts/tunedXgboost.pkl", "wb") as f:
    pickle.dump(model_package, f)